In [1]:
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.types import Integer, String, Date, BigInteger

metro = pd.read_csv("D:\\Diplomado\\Modulo 4\\PROYECTO FINAL\\afluenciastc_desglosado_04_2026.csv")

# Asegurar formato de fecha
metro['fecha'] = pd.to_datetime(metro['fecha'])

# =========================
# 1. Dimensión Fecha
# =========================

dim_fecha = metro[['fecha', 'mes', 'anio']].drop_duplicates().copy()
dim_fecha['id_fecha'] = dim_fecha['fecha'].dt.strftime('%Y%m%d').astype(int)
dim_fecha['dia'] = dim_fecha['fecha'].dt.day
dim_fecha['trimestre'] = dim_fecha['fecha'].dt.quarter
dim_fecha['dia_semana'] = dim_fecha['fecha'].dt.day_name()
dim_fecha = dim_fecha[['id_fecha', 'fecha', 'dia', 'mes', 'trimestre', 'anio', 'dia_semana']]

# =========================
# 2. Dimensión Estación
# =========================

dim_estacion = metro[['linea', 'estacion']].drop_duplicates().copy()
dim_estacion = (dim_estacion.sort_values(['linea', 'estacion']).reset_index(drop=True))
dim_estacion['id_estacion'] = dim_estacion.index + 1
dim_estacion = dim_estacion[['id_estacion', 'linea', 'estacion']]

# =========================
# 3. Dimensión Tipo de Pago
# =========================

dim_tipo_pago = metro[['tipo_pago']].drop_duplicates().copy()
dim_tipo_pago = (dim_tipo_pago.sort_values('tipo_pago').reset_index(drop=True))
dim_tipo_pago['id_tipo_pago'] = dim_tipo_pago.index + 1
dim_tipo_pago = dim_tipo_pago[['id_tipo_pago', 'tipo_pago']]

# =========================
# 4. Tabla de hechos
# =========================

metro_fact = metro.copy()
metro_fact['id_fecha'] = metro_fact['fecha'].dt.strftime('%Y%m%d').astype(int)
metro_fact = metro_fact.merge(dim_estacion,on=['linea', 'estacion'],how='left')
metro_fact = metro_fact.merge(dim_tipo_pago,on='tipo_pago',how='left')
fact_afluencia = metro_fact[['id_fecha', 'id_estacion', 'id_tipo_pago', 'afluencia']].copy()


AURORA_HOST     = "aurora-mod4-instance-1.czk5u3frnmxe.us-east-1.rds.amazonaws.com"
AURORA_PASSWORD = "Laisaa30_"
AURORA_DATABASE = "northwind"

engine = create_engine(f"postgresql+psycopg2://postgres:{AURORA_PASSWORD}@{AURORA_HOST}:5432/{AURORA_DATABASE}")

# Creacion del esquema
with engine.begin() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS metro_dwh_py"))

print("✓ schema metro_dwh_py listo")

# Tipos de dato por tabla
dtype_dim_fecha = {"id_fecha": Integer(), "fecha": Date(), "dia": Integer(), "mes": String(20), "trimestre": Integer(), "anio": Integer(), "dia_semana": String(20)}
dtype_dim_estacion = {"id_estacion": Integer(), "linea": String(50), "estacion": String(100)}
dtype_dim_tipo_pago = {"id_tipo_pago": Integer(), "tipo_pago": String(50)}
dtype_fact_afluencia = {"id_fecha": Integer(), "id_estacion": Integer(), "id_tipo_pago": Integer(), "afluencia": BigInteger()}

# Carga ordenada de las 5 dims + fact_sales
tablas_en_orden = [
    ("dim_fecha", dim_fecha, dtype_dim_fecha),
    ("dim_estacion", dim_estacion, dtype_dim_estacion),
    ("dim_tipo_pago", dim_tipo_pago, dtype_dim_tipo_pago),
    ("fact_afluencia", fact_afluencia, dtype_fact_afluencia)
]

for nombre, df, dtypes in tablas_en_orden:
    df.to_sql(name=nombre,con=engine,schema="metro_dwh_py",if_exists="replace",index=False,chunksize=1000,method="multi",dtype=dtypes)
    print(f"✓ {nombre:14s} {len(df):>5} filas")

print("\nCarga completa.")

✓ schema metro_dwh_py listo
✓ dim_fecha       1946 filas
✓ dim_estacion     195 filas
✓ dim_tipo_pago      3 filas
✓ fact_afluencia 1138410 filas

Carga completa.


In [3]:
# ============================================================
# 5. GENERACIÓN DE CONSULTAS OLAP + GRÁFICAS PLOTLY
# ============================================================

import os
import plotly.express as px

# Carpeta donde se guardarán las gráficas HTML
os.makedirs("graficas", exist_ok=True)

# Esquema de la base OLAP
SCHEMA = "metro_dwh_py"

# ============================================================
# COLORES Y FORMATO DEL DASHBOARD
# ============================================================

colores_lineas = {
    "Linea 1": "#E94798",
    "Linea 2": "#005BAC",
    "Linea 3": "#B5A000",
    "Linea 4": "#65B8AE",
    "Linea 5": "#FFD200",
    "Linea 6": "#E41E1B",
    "Linea 7": "#F37021",
    "Linea 8": "#009A44",
    "Linea 9": "#4B2E2A",
    "Linea A": "#A51890",
    "Linea B": "#A7A9AC",
    "Linea 12": "#C19A5B"
}

color_naranja = "#F37021"
color_fondo = "#0B0B0B"
color_grafica = "#111111"
color_texto = "#FFFFFF"

layout_dark = dict(
    plot_bgcolor=color_grafica,
    paper_bgcolor=color_fondo,
    font=dict(color=color_texto, family="Arial"),
    title_x=0.5,
    title_font=dict(size=22),
    margin=dict(l=60, r=40, t=80, b=60)
)

escala_naranja_dark = [
    [0.0, "#111111"],
    [0.35, "#4B2E2A"],
    [0.70, "#A84600"],
    [1.0, "#F37021"]
]

escala_rojo_azul = "RdBu_r"


def guardar_figura(fig, nombre):
    fig.write_html(
        f"graficas/{nombre}.html",
        include_plotlyjs="cdn",
        full_html=True
    )
    print(f"✓ Guardada: graficas/{nombre}.html")


def leer_sql(query):
    return pd.read_sql_query(text(query), con=engine)


# ============================================================
# CONSULTA A. AFLUENCIA MENSUAL TOTAL
# Gráficas 1 y 6
# ============================================================

sql_mensual_total = f"""
SELECT
    TO_CHAR(d.fecha, 'YYYY-MM') AS anio_mes,
    d.anio,
    EXTRACT(MONTH FROM d.fecha)::INT AS mes_num,
    SUM(f.afluencia) AS afluencia
FROM {SCHEMA}.fact_afluencia f
INNER JOIN {SCHEMA}.dim_fecha d
    ON f.id_fecha = d.id_fecha
GROUP BY
    TO_CHAR(d.fecha, 'YYYY-MM'),
    d.anio,
    EXTRACT(MONTH FROM d.fecha)::INT
ORDER BY
    anio_mes;
"""

mensual_total = leer_sql(sql_mensual_total)


# ============================================================
# CONSULTA B. AFLUENCIA MENSUAL POR LÍNEA
# Gráficas 2 y 9
# ============================================================

sql_mensual_linea = f"""
SELECT
    TO_CHAR(d.fecha, 'YYYY-MM') AS anio_mes,
    d.anio,
    EXTRACT(MONTH FROM d.fecha)::INT AS mes_num,
    e.linea,
    SUM(f.afluencia) AS afluencia
FROM {SCHEMA}.fact_afluencia f
INNER JOIN {SCHEMA}.dim_fecha d
    ON f.id_fecha = d.id_fecha
INNER JOIN {SCHEMA}.dim_estacion e
    ON f.id_estacion = e.id_estacion
GROUP BY
    TO_CHAR(d.fecha, 'YYYY-MM'),
    d.anio,
    EXTRACT(MONTH FROM d.fecha)::INT,
    e.linea
ORDER BY
    anio_mes,
    e.linea;
"""

mensual_linea = leer_sql(sql_mensual_linea)


# ============================================================
# CONSULTA C. AFLUENCIA DIARIA POR LÍNEA
# Gráficas 3 y 7
# ============================================================

sql_diario_linea = f"""
SELECT
    d.fecha,
    e.linea,
    SUM(f.afluencia) AS afluencia
FROM {SCHEMA}.fact_afluencia f
INNER JOIN {SCHEMA}.dim_fecha d
    ON f.id_fecha = d.id_fecha
INNER JOIN {SCHEMA}.dim_estacion e
    ON f.id_estacion = e.id_estacion
GROUP BY
    d.fecha,
    e.linea
ORDER BY
    d.fecha,
    e.linea;
"""

diario_linea = leer_sql(sql_diario_linea)


# ============================================================
# CONSULTA D. PROMEDIO DIARIO POR LÍNEA Y DÍA DE SEMANA
# Gráfica 4
# ============================================================

sql_heat_linea_dia = f"""
WITH diario_linea AS (
    SELECT
        d.fecha,
        d.dia_semana,
        e.linea,
        SUM(f.afluencia) AS afluencia_diaria
    FROM {SCHEMA}.fact_afluencia f
    INNER JOIN {SCHEMA}.dim_fecha d
        ON f.id_fecha = d.id_fecha
    INNER JOIN {SCHEMA}.dim_estacion e
        ON f.id_estacion = e.id_estacion
    GROUP BY
        d.fecha,
        d.dia_semana,
        e.linea
)
SELECT
    linea,
    dia_semana,
    AVG(afluencia_diaria) AS afluencia
FROM diario_linea
GROUP BY
    linea,
    dia_semana
ORDER BY
    linea,
    dia_semana;
"""

heat_linea_dia = leer_sql(sql_heat_linea_dia)


# ============================================================
# CONSULTA E. AFLUENCIA DIARIA POR ESTACIÓN
# Gráfica 5
# ============================================================

sql_diario_estacion = f"""
SELECT
    d.fecha,
    e.linea,
    e.estacion,
    CONCAT(e.linea, ' - ', e.estacion) AS estacion_completa,
    SUM(f.afluencia) AS afluencia
FROM {SCHEMA}.fact_afluencia f
INNER JOIN {SCHEMA}.dim_fecha d
    ON f.id_fecha = d.id_fecha
INNER JOIN {SCHEMA}.dim_estacion e
    ON f.id_estacion = e.id_estacion
GROUP BY
    d.fecha,
    e.linea,
    e.estacion
ORDER BY
    d.fecha,
    e.linea,
    e.estacion;
"""

diario_estacion = leer_sql(sql_diario_estacion)


# ============================================================
# CONSULTA F. TOTAL POR LÍNEA
# Gráfica 8
# ============================================================

sql_total_linea = f"""
SELECT
    e.linea,
    SUM(f.afluencia) AS afluencia
FROM {SCHEMA}.fact_afluencia f
INNER JOIN {SCHEMA}.dim_estacion e
    ON f.id_estacion = e.id_estacion
GROUP BY
    e.linea
ORDER BY
    afluencia DESC;
"""

total_linea = leer_sql(sql_total_linea)


# ============================================================
# CONSULTA G. AFLUENCIA DIARIA TOTAL
# Gráficas 10, 11 y 12
# ============================================================

sql_diario_total = f"""
SELECT
    d.fecha,
    d.dia_semana,
    SUM(f.afluencia) AS afluencia
FROM {SCHEMA}.fact_afluencia f
INNER JOIN {SCHEMA}.dim_fecha d
    ON f.id_fecha = d.id_fecha
GROUP BY
    d.fecha,
    d.dia_semana
ORDER BY
    d.fecha;
"""

diario_total = leer_sql(sql_diario_total)


# ============================================================
# LIMPIEZA AUXILIAR PARA GRÁFICAS
# ============================================================

mapa_dias = {
    "Monday": "Lunes",
    "Tuesday": "Martes",
    "Wednesday": "Miércoles",
    "Thursday": "Jueves",
    "Friday": "Viernes",
    "Saturday": "Sábado",
    "Sunday": "Domingo",
    "Lunes": "Lunes",
    "Martes": "Martes",
    "Miércoles": "Miércoles",
    "Jueves": "Jueves",
    "Viernes": "Viernes",
    "Sábado": "Sábado",
    "Domingo": "Domingo"
}

orden_dias = [
    "Lunes", "Martes", "Miércoles",
    "Jueves", "Viernes", "Sábado", "Domingo"
]

heat_linea_dia["dia_semana"] = heat_linea_dia["dia_semana"].map(mapa_dias)
diario_total["dia_semana"] = diario_total["dia_semana"].map(mapa_dias)

diario_linea["fecha"] = pd.to_datetime(diario_linea["fecha"])
diario_estacion["fecha"] = pd.to_datetime(diario_estacion["fecha"])
diario_total["fecha"] = pd.to_datetime(diario_total["fecha"])


# ============================================================
# GRÁFICA 1. AFLUENCIA MENSUAL TOTAL
# ============================================================

fig = px.line(
    mensual_total,
    x="anio_mes",
    y="afluencia",
    markers=True,
    title="Afluencia mensual total del Metro"
)

fig.update_traces(
    line=dict(color=color_naranja, width=4),
    marker=dict(color=color_naranja, size=7)
)

fig.update_layout(
    **layout_dark,
    xaxis_title="Mes",
    yaxis_title="Usuarios"
)

guardar_figura(fig, "01_afluencia_mensual_total")


# ============================================================
# GRÁFICA 2. AFLUENCIA MENSUAL POR LÍNEA
# ============================================================

fig = px.line(
    mensual_linea,
    x="anio_mes",
    y="afluencia",
    color="linea",
    markers=True,
    title="Afluencia mensual por línea",
    color_discrete_map=colores_lineas
)

fig.update_layout(
    **layout_dark,
    xaxis_title="Mes",
    yaxis_title="Usuarios",
    legend_title="Línea"
)

guardar_figura(fig, "02_afluencia_mensual_por_linea")


# ============================================================
# GRÁFICA 3. PROMEDIO DIARIO DE USUARIOS POR LÍNEA
# ============================================================

promedio_linea = (
    diario_linea
    .groupby("linea", as_index=False)["afluencia"]
    .mean()
    .sort_values("afluencia", ascending=True)
)

fig = px.bar(
    promedio_linea,
    x="afluencia",
    y="linea",
    orientation="h",
    color="linea",
    color_discrete_map=colores_lineas,
    text="afluencia",
    title="Promedio diario de usuarios por línea"
)

fig.update_traces(
    texttemplate="%{text:,.0f}",
    textposition="outside"
)

fig.update_layout(
    **layout_dark,
    xaxis_title="Usuarios promedio diarios",
    yaxis_title="Línea",
    showlegend=False
)

guardar_figura(fig, "03_promedio_diario_por_linea")


# ============================================================
# GRÁFICA 4. HEATMAP LÍNEA VS DÍA DE LA SEMANA
# ============================================================

heat_linea_dia["dia_semana"] = pd.Categorical(
    heat_linea_dia["dia_semana"],
    categories=orden_dias,
    ordered=True
)

tabla_heat = heat_linea_dia.pivot(
    index="linea",
    columns="dia_semana",
    values="afluencia"
)

tabla_heat = tabla_heat[orden_dias]

fig = px.imshow(
    tabla_heat,
    text_auto=".0f",
    aspect="auto",
    title="Promedio de usuarios por línea y día de la semana",
    color_continuous_scale=escala_rojo_azul
)

fig.update_layout(
    **layout_dark,
    xaxis_title="Día de la semana",
    yaxis_title="Línea"
)

guardar_figura(fig, "04_heatmap_linea_dia")


# ============================================================
# GRÁFICA 5. TOP 15 ESTACIONES
# ============================================================

top_estaciones = (
    diario_estacion
    .groupby("estacion_completa", as_index=False)["afluencia"]
    .mean()
    .sort_values("afluencia", ascending=False)
    .head(15)
    .sort_values("afluencia", ascending=True)
)

fig = px.bar(
    top_estaciones,
    x="afluencia",
    y="estacion_completa",
    orientation="h",
    text="afluencia",
    title="Top 15 estaciones con mayor afluencia promedio diaria"
)

fig.update_traces(
    marker_color=color_naranja,
    texttemplate="%{text:,.0f}",
    textposition="outside"
)

fig.update_layout(
    **layout_dark,
    xaxis_title="Usuarios promedio diarios",
    yaxis_title="Estación"
)

guardar_figura(fig, "05_top_estaciones")


# ============================================================
# GRÁFICA 6. HEATMAP AÑO VS MES
# ============================================================

tabla_anio_mes = mensual_total.pivot(
    index="anio",
    columns="mes_num",
    values="afluencia"
)

fig = px.imshow(
    tabla_anio_mes,
    text_auto=".2s",
    aspect="auto",
    title="Afluencia total por año y mes",
    color_continuous_scale=escala_naranja_dark
)

fig.update_layout(
    **layout_dark,
    xaxis_title="Mes",
    yaxis_title="Año"
)

guardar_figura(fig, "06_heatmap_anio_mes")


# ============================================================
# GRÁFICA 7. BOXPLOT DE AFLUENCIA DIARIA POR LÍNEA
# ============================================================

fig = px.box(
    diario_linea,
    x="linea",
    y="afluencia",
    color="linea",
    color_discrete_map=colores_lineas,
    title="Distribución de afluencia diaria por línea"
)

fig.update_layout(
    **layout_dark,
    xaxis_title="Línea",
    yaxis_title="Usuarios diarios",
    showlegend=False
)

guardar_figura(fig, "07_boxplot_linea")


# ============================================================
# GRÁFICA 8. TREEMAP DE PARTICIPACIÓN POR LÍNEA
# ============================================================

fig = px.treemap(
    total_linea,
    path=["linea"],
    values="afluencia",
    color="linea",
    color_discrete_map=colores_lineas,
    title="Participación de afluencia total por línea"
)

fig.update_layout(
    **layout_dark
)

guardar_figura(fig, "08_treemap_linea")


# ============================================================
# GRÁFICA 9. MOSAICO DE EVOLUCIÓN MENSUAL POR LÍNEA
# ============================================================

fig = px.line(
    mensual_linea,
    x="anio_mes",
    y="afluencia",
    color="linea",
    facet_col="linea",
    facet_col_wrap=3,
    color_discrete_map=colores_lineas,
    title="Evolución mensual de afluencia por línea"
)

fig.update_layout(
    **layout_dark,
    height=1000,
    showlegend=False
)

fig.update_xaxes(matches=None, showticklabels=True)
fig.update_yaxes(matches=None, showticklabels=True)

guardar_figura(fig, "09_mosaico_linea")


# ============================================================
# GRÁFICA 10. AFLUENCIA DIARIA TOTAL
# ============================================================

fig = px.line(
    diario_total,
    x="fecha",
    y="afluencia",
    title="Afluencia diaria total del Metro"
)

fig.update_traces(
    line=dict(color=color_naranja, width=2)
)

fig.update_layout(
    **layout_dark,
    xaxis_title="Fecha",
    yaxis_title="Usuarios"
)

guardar_figura(fig, "10_afluencia_diaria_total")


# ============================================================
# GRÁFICA 11. PROMEDIO POR DÍA DE LA SEMANA
# ============================================================

promedio_dia = (
    diario_total
    .groupby("dia_semana", as_index=False)["afluencia"]
    .mean()
)

promedio_dia["dia_semana"] = pd.Categorical(
    promedio_dia["dia_semana"],
    categories=orden_dias,
    ordered=True
)

promedio_dia = promedio_dia.sort_values("dia_semana")

fig = px.bar(
    promedio_dia,
    x="dia_semana",
    y="afluencia",
    text="afluencia",
    title="Promedio de afluencia por día de la semana"
)

fig.update_traces(
    marker_color=color_naranja,
    texttemplate="%{text:,.0f}",
    textposition="outside"
)

fig.update_layout(
    **layout_dark,
    xaxis_title="Día de la semana",
    yaxis_title="Usuarios promedio"
)

guardar_figura(fig, "11_promedio_dia_semana")


# ============================================================
# GRÁFICA 12. DÍAS ATÍPICOS
# ============================================================

media = diario_total["afluencia"].mean()
desv = diario_total["afluencia"].std()

limite_superior = media + 2 * desv
limite_inferior = media - 2 * desv

diario_total["tipo"] = "Normal"
diario_total.loc[diario_total["afluencia"] > limite_superior, "tipo"] = "Muy alto"
diario_total.loc[diario_total["afluencia"] < limite_inferior, "tipo"] = "Muy bajo"

colores_atipicos = {
    "Normal": color_naranja,
    "Muy alto": "#00FF88",
    "Muy bajo": "#FF3B3B"
}

fig = px.scatter(
    diario_total,
    x="fecha",
    y="afluencia",
    color="tipo",
    color_discrete_map=colores_atipicos,
    title="Días atípicos de afluencia total"
)

fig.add_hline(
    y=limite_superior,
    line_dash="dash",
    line_color="#00FF88"
)

fig.add_hline(
    y=limite_inferior,
    line_dash="dash",
    line_color="#FF3B3B"
)

fig.update_layout(
    **layout_dark,
    xaxis_title="Fecha",
    yaxis_title="Usuarios",
    legend_title="Tipo de día"
)

guardar_figura(fig, "12_dias_atipicos")


print("\n✓ Todas las consultas OLAP se ejecutaron correctamente.")
print("✓ Todas las gráficas se guardaron en la carpeta 'graficas'.")

✓ Guardada: graficas/01_afluencia_mensual_total.html
✓ Guardada: graficas/02_afluencia_mensual_por_linea.html
✓ Guardada: graficas/03_promedio_diario_por_linea.html
✓ Guardada: graficas/04_heatmap_linea_dia.html
✓ Guardada: graficas/05_top_estaciones.html
✓ Guardada: graficas/06_heatmap_anio_mes.html
✓ Guardada: graficas/07_boxplot_linea.html
✓ Guardada: graficas/08_treemap_linea.html
✓ Guardada: graficas/09_mosaico_linea.html
✓ Guardada: graficas/10_afluencia_diaria_total.html
✓ Guardada: graficas/11_promedio_dia_semana.html
✓ Guardada: graficas/12_dias_atipicos.html

✓ Todas las consultas OLAP se ejecutaron correctamente.
✓ Todas las gráficas se guardaron en la carpeta 'graficas'.
